# CATfusion on TCGA-BRCA (Colab)

Runs the heavy steps on a Colab GPU: slide features (step 3) and paper-scale training (step 5).
Data and results are kept in Google Drive, so a disconnected session can pick up where it stopped:
every step skips work that is already finished.

**Before you start:** Runtime → Change runtime type → GPU. Put `ctranspath.pth`
(from github.com/Xiyue-Wang/TransPath) in `My Drive/capstone/checkpoints/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/capstone'
!mkdir -p $DRIVE/data $DRIVE/results $DRIVE/checkpoints

In [ ]:
!git clone -q https://github.com/alexia-moreanu/Capstone.git /content/Capstone || git -C /content/Capstone pull -q
%cd /content/Capstone
!git checkout -q catfusion-jax-replication
# keep data/results on Drive
!rm -rf data results checkpoints && ln -s $DRIVE/data data && ln -s $DRIVE/results results && ln -s $DRIVE/checkpoints checkpoints
!pip -q install -r requirements.txt -r requirements-slides.txt "jax[cuda12]"
!apt-get -qq install -y openslide-tools > /dev/null
!python -m pytest -q tests

## 1. Clinical + omics → TabAE tokens (no slides; a few GB)

In [ ]:
!./scripts/run_tcga.sh omics

## 2. Slides: pilot first, then everything
Each slide is downloaded, turned into ~100–300 mosaic features, and deleted. Re-run the cell after a disconnect.

In [ ]:
!DEVICE=cuda DOWNLOAD_DIR=/content/tmp_slides ./scripts/run_tcga.sh slides-pilot 20

In [ ]:
!DEVICE=cuda DOWNLOAD_DIR=/content/tmp_slides ./scripts/run_tcga.sh slides

## 3. Cohort, experiments, report

In [ ]:
!./scripts/run_tcga.sh cohort

In [ ]:
# SQ1/SQ2 with the released recipe (lr 1e-5, weight decay 0.7, 100 epochs, 5 repeats).
# Start with REPEATS=1 to time one repeat.
!REPEATS=1 ./scripts/run_tcga.sh train replication

In [ ]:
!./scripts/run_tcga.sh train fusion
!./scripts/run_tcga.sh train protocol
!./scripts/run_tcga.sh train extension
!./scripts/run_tcga.sh baselines

In [ ]:
!./scripts/run_tcga.sh report
from IPython.display import Markdown
Markdown(open('results/brca/report.md').read())